# Deep Learning Diagnostics Lab

작은 CNN 하나를 학습시키면서 다음 흐름을 따라갑니다.

```text
학습이 잘 되는가?
→ 어느 층이 실제로 움직이는가?
→ 내부 표현이 몇 개 방향을 쓰는가?
→ class 정보가 표현 안에 잘 정리되는가?
→ 표현 구조가 학습 중 언제 바뀌는가?
→ 표현 공간의 모양과 국소 구조는 어떤가?
→ parameter 공간의 loss surface는 어떤가?
```

사용 도구: loss/accuracy, gradient norm, update-to-weight ratio, effective rank, linear probe, CKA, PCA/UMAP, local PCA+kNN, Hessian top eigenvalue, weight interpolation.

CIFAR-10은 `torchvision` 원본 서버 대신 **Hugging Face Hub (`uoft-cs/cifar10`)** 에서 받습니다.

## 0. 환경 설정

`FAST_MODE=True`이면 CIFAR-10 일부와 6 epoch만 사용합니다.

CSV/NPZ/PNG/TensorBoard/JSON은 Google Drive에 저장하고, model weight는 Colab runtime에만 둡니다.

In [ ]:
!pip -q install datasets umap-learn tensorboard

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import umap.umap_ as umap

from datasets import load_dataset
from google.colab import drive
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
SEED = 7
FAST_MODE = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount("/content/drive")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/deep_learning_diagnostics"
)
CSV_DIR = DRIVE_ROOT / "csv"
NPZ_DIR = DRIVE_ROOT / "npz"
FIG_DIR = DRIVE_ROOT / "figures"
TB_DIR = DRIVE_ROOT / "tensorboard"
SUMMARY_DIR = DRIVE_ROOT / "summaries"

LOCAL_CKPT_DIR = Path("/content/local_checkpoints")

for folder in [
    CSV_DIR,
    NPZ_DIR,
    FIG_DIR,
    TB_DIR,
    SUMMARY_DIR,
    LOCAL_CKPT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

## 1. 데이터와 진단 시점 준비

학습용 입력에는 crop/flip augmentation을 적용합니다.

반면 CKA, PCA처럼 epoch 사이 representation을 비교할 때는 **같은 이미지를 같은 순서로 넣어야** 하므로 진단용 입력에는 augmentation을 적용하지 않습니다.

- 6 epochs → `0, 2, 4, 6`
- 12 epochs → `0, 3, 6, 9, 12`

여기서 epoch 0은 학습 전 초기 상태입니다.

In [ ]:
mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

print("Loading CIFAR-10 from Hugging Face Hub...")
hf_train = load_dataset(
    "uoft-cs/cifar10",
    split="train",
)
print("HF cache ready:", len(hf_train), "samples")


class CIFAR10FromHF(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        image = row["img"].convert("RGB")
        label = int(row["label"])

        return self.transform(image), label


train_aug = CIFAR10FromHF(
    hf_train,
    train_transform,
)
train_eval = CIFAR10FromHF(
    hf_train,
    eval_transform,
)

permutation = torch.randperm(
    len(train_aug),
    generator=torch.Generator().manual_seed(SEED),
).tolist()

if FAST_MODE:
    n_train = 12_000
    n_val = 2_000
    EPOCHS = 6
else:
    n_train = 40_000
    n_val = 5_000
    EPOCHS = 12

train_indices = permutation[:n_train]
val_indices = permutation[n_train : n_train + n_val]

train_dataset = Subset(
    train_aug,
    train_indices,
)
train_eval_dataset = Subset(
    train_eval,
    train_indices,
)
val_dataset = Subset(
    train_eval,
    val_indices,
)

loader_args = {
    "batch_size": 256,
    "num_workers": 2,
    "pin_memory": torch.cuda.is_available(),
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_args,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    shuffle=False,
    **loader_args,
)
val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_args,
)

diag_interval = max(1, round(EPOCHS / 4))
DIAG_EPOCHS = sorted(
    set(
        [0]
        + list(range(diag_interval, EPOCHS + 1, diag_interval))
        + [EPOCHS]
    )
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Diagnostic epochs:", DIAG_EPOCHS)

## 2. 진단할 CNN 정의

```text
image → stem → block1 → block2 → penultimate → head → logits
```

중간 convolution 출력은 `(channel, height, width)` 형태이므로 공간 평균을 내서 sample 하나당 feature vector 하나로 바꿉니다.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
        )

        self.block1 = nn.Sequential(
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.penultimate = nn.Linear(128, 64)
        self.head = nn.Linear(64, 10)

    def forward(self, x, return_features=False):
        features = {}

        x = self.stem(x)
        features["stem"] = x.mean(dim=(2, 3))

        x = self.block1(x)
        features["block1"] = x.mean(dim=(2, 3))

        x = self.block2(x)
        features["block2"] = x.mean(dim=(2, 3))

        x = self.pool(x).flatten(1)
        x = F.relu(self.penultimate(x))
        features["penultimate"] = x

        logits = self.head(x)

        if return_features:
            return logits, features

        return logits


LAYER_NAMES = [
    "stem",
    "block1",
    "block2",
    "penultimate",
]

TRACKED_PARAMETERS = {
    "stem": "stem.0.weight",
    "block1": "block1.0.weight",
    "block2": "block2.0.weight",
    "penultimate": "penultimate.weight",
    "head": "head.weight",
}

# SGD와 AdamW를 같은 초기값에서 시작시킵니다.
torch.manual_seed(SEED)
INITIAL_STATE = {
    name: value.cpu().clone()
    for name, value in SmallCNN().state_dict().items()
}

## 3. 기본 평가 함수

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)

        total_loss += F.cross_entropy(
            logits,
            y,
            reduction="sum",
        ).item()

        total_correct += (
            logits.argmax(dim=1) == y
        ).sum().item()
        total_count += y.numel()

    average_loss = total_loss / total_count
    accuracy = total_correct / total_count

    return average_loss, accuracy

## 4. 학습하면서 gradient와 실제 parameter 이동 기록

매 batch에서 다음 두 값을 기록합니다.

1. `gradient norm = ||dL/dW||`
2. `update-to-weight = ||ΔW|| / ||W||`

Gradient가 커도 optimizer 때문에 실제 parameter 이동이 작을 수 있으므로 둘을 같이 봅니다.

In [ ]:
def make_optimizer(model, optimizer_name):
    if optimizer_name == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=0.08,
            momentum=0.9,
        )

    if optimizer_name == "adamw":
        return torch.optim.AdamW(
            model.parameters(),
            lr=2e-3,
        )

    raise ValueError(optimizer_name)


def train_model(run_name, optimizer_name):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(INITIAL_STATE)

    optimizer = make_optimizer(
        model,
        optimizer_name,
    )
    writer = SummaryWriter(
        str(TB_DIR / run_name)
    )

    history_rows = []
    dynamics_rows = []
    global_step = 0

    torch.save(
        model.state_dict(),
        LOCAL_CKPT_DIR / f"{run_name}_epoch0.pt",
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_count = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()

            named_parameters = dict(
                model.named_parameters()
            )

            # optimizer.step() 전 값을 복사해 실제 ΔW를 계산합니다.
            before_update = {
                tag: named_parameters[param_name]
                .detach()
                .clone()
                for tag, param_name in TRACKED_PARAMETERS.items()
            }

            dynamics_row = {
                "run": run_name,
                "epoch": epoch,
                "step": global_step,
            }

            for tag, param_name in TRACKED_PARAMETERS.items():
                parameter = named_parameters[param_name]
                grad_norm = parameter.grad.detach().norm().item()

                dynamics_row[f"{tag}_grad_norm"] = grad_norm

            optimizer.step()

            named_parameters = dict(
                model.named_parameters()
            )

            for tag, param_name in TRACKED_PARAMETERS.items():
                parameter = named_parameters[param_name]

                delta_norm = (
                    parameter.detach() - before_update[tag]
                ).norm().item()
                weight_norm = parameter.detach().norm().item()

                update_ratio = delta_norm / (
                    weight_norm + 1e-12
                )

                dynamics_row[
                    f"{tag}_update_to_weight"
                ] = update_ratio

                writer.add_scalar(
                    f"gradient/{tag}",
                    dynamics_row[f"{tag}_grad_norm"],
                    global_step,
                )
                writer.add_scalar(
                    f"update_to_weight/{tag}",
                    update_ratio,
                    global_step,
                )

            dynamics_rows.append(dynamics_row)
            global_step += 1

            train_loss_sum += loss.item() * y.numel()
            train_correct += (
                logits.argmax(dim=1) == y
            ).sum().item()
            train_count += y.numel()

        val_loss, val_accuracy = evaluate(
            model,
            val_loader,
        )

        history_row = {
            "run": run_name,
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "train_accuracy": train_correct / train_count,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
        history_rows.append(history_row)

        print(run_name, history_row)

        writer.add_scalar(
            "metric/val_loss",
            val_loss,
            epoch,
        )
        writer.add_scalar(
            "metric/val_accuracy",
            val_accuracy,
            epoch,
        )

        if epoch in DIAG_EPOCHS:
            torch.save(
                model.state_dict(),
                LOCAL_CKPT_DIR
                / f"{run_name}_epoch{epoch}.pt",
            )

    writer.close()

    history_df = pd.DataFrame(history_rows)
    dynamics_df = pd.DataFrame(dynamics_rows)

    history_df.to_csv(
        CSV_DIR / f"{run_name}_history.csv",
        index=False,
    )
    dynamics_df.to_csv(
        CSV_DIR / f"{run_name}_dynamics.csv",
        index=False,
    )

    return model, history_df, dynamics_df


sgd_model, sgd_history, sgd_dynamics = train_model(
    "sgd",
    "sgd",
)

adamw_model, adamw_history, adamw_dynamics = train_model(
    "adamw",
    "adamw",
)

## 5. 고정 입력 representation 추출

각 진단 checkpoint에서 같은 이미지들을 같은 순서로 넣고 각 layer feature를 저장합니다.

In [ ]:
def load_checkpoint(run_name, epoch):
    path = LOCAL_CKPT_DIR / (
        f"{run_name}_epoch{epoch}.pt"
    )
    return torch.load(
        path,
        map_location="cpu",
    )


@torch.no_grad()
def collect_features(
    state_dict,
    loader,
    max_samples=2000,
):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    feature_buckets = {
        layer: []
        for layer in LAYER_NAMES
    }
    label_bucket = []
    seen = 0

    for x, y in loader:
        x = x.to(DEVICE)

        _, features = model(
            x,
            return_features=True,
        )

        take = min(
            x.size(0),
            max_samples - seen,
        )

        for layer in LAYER_NAMES:
            feature_buckets[layer].append(
                features[layer][:take].cpu()
            )

        label_bucket.append(y[:take].cpu())
        seen += take

        if seen >= max_samples:
            break

    feature_arrays = {
        layer: torch.cat(parts).numpy()
        for layer, parts in feature_buckets.items()
    }
    labels = torch.cat(label_bucket).numpy()

    return feature_arrays, labels


FEATURE_CACHE = {}

for run_name in ["sgd", "adamw"]:
    for epoch in DIAG_EPOCHS:
        state_dict = load_checkpoint(
            run_name,
            epoch,
        )
        FEATURE_CACHE[(run_name, epoch)] = collect_features(
            state_dict,
            train_eval_loader,
        )

print(
    "Feature snapshots:",
    list(FEATURE_CACHE.keys()),
)

## 6. Effective rank

SVD spectrum을 entropy 형태로 요약해 각 layer가 실질적으로 몇 개 방향을 사용하는지 봅니다.

In [ ]:
def effective_rank(features):
    centered = features - features.mean(
        axis=0,
        keepdims=True,
    )

    singular_values = np.linalg.svd(
        centered,
        compute_uv=False,
    )

    eigenvalues = singular_values ** 2
    probabilities = eigenvalues / (
        eigenvalues.sum() + 1e-12
    )

    entropy = -(
        probabilities
        * np.log(probabilities + 1e-12)
    ).sum()

    return float(np.exp(entropy))


rank_rows = []

for (run_name, epoch), (features, labels) in FEATURE_CACHE.items():
    for layer_name, layer_features in features.items():
        rank_rows.append(
            {
                "run": run_name,
                "epoch": epoch,
                "layer": layer_name,
                "effective_rank": effective_rank(
                    layer_features
                ),
            }
        )

rank_df = pd.DataFrame(rank_rows)
rank_df.to_csv(
    CSV_DIR / "effective_rank.csv",
    index=False,
)

display(rank_df)

## 7. Linear probe

원래 CNN은 고정하고 각 layer feature 위에 선형 분류기만 학습합니다.

Probe accuracy가 높을수록 그 layer 표현에서 class 정보를 선형적으로 읽기 쉽다는 뜻입니다.

In [ ]:
probe_rows = []

for epoch in DIAG_EPOCHS:
    features, labels = FEATURE_CACHE[("sgd", epoch)]

    for layer_name in LAYER_NAMES:
        x = features[layer_name]
        split_index = int(len(x) * 0.8)

        x_train = x[:split_index]
        y_train = labels[:split_index]
        x_test = x[split_index:]
        y_test = labels[split_index:]

        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(x_train)
        x_test_scaled = scaler.transform(x_test)

        classifier = LogisticRegression(
            max_iter=500,
            n_jobs=-1,
        )
        classifier.fit(
            x_train_scaled,
            y_train,
        )

        accuracy = classifier.score(
            x_test_scaled,
            y_test,
        )

        probe_rows.append(
            {
                "epoch": epoch,
                "layer": layer_name,
                "probe_accuracy": accuracy,
            }
        )

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(
    CSV_DIR / "linear_probe.csv",
    index=False,
)

display(probe_df)

## 8. CKA

각 layer의 학습 전 표현과 학습 중 표현이 sample 관계 관점에서 얼마나 비슷한지 봅니다.

In [ ]:
def linear_cka(x, y):
    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)

    cross_covariance = x.T @ y

    numerator = (
        cross_covariance * cross_covariance
    ).sum()

    denominator = np.sqrt(
        ((x.T @ x) ** 2).sum()
        * ((y.T @ y) ** 2).sum()
        + 1e-12
    )

    return float(numerator / denominator)


cka_rows = []

for layer_name in LAYER_NAMES:
    initial_features = FEATURE_CACHE[("sgd", 0)][0][
        layer_name
    ]

    for epoch in DIAG_EPOCHS:
        current_features = FEATURE_CACHE[("sgd", epoch)][0][
            layer_name
        ]

        cka_value = linear_cka(
            initial_features,
            current_features,
        )

        cka_rows.append(
            {
                "layer": layer_name,
                "epoch": epoch,
                "cka_to_init": cka_value,
            }
        )

cka_df = pd.DataFrame(cka_rows)
cka_df.to_csv(
    CSV_DIR / "cka_to_init.csv",
    index=False,
)

display(cka_df)

## 9. PCA와 UMAP

최종 penultimate representation을 2차원으로 내려 class별 점구름을 시각화합니다.

In [ ]:
final_features, final_labels = FEATURE_CACHE[(
    "sgd",
    EPOCHS,
)]

penultimate_features = final_features[
    "penultimate"
]

pca_xy = PCA(
    n_components=2
).fit_transform(
    penultimate_features
)

umap_xy = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.1,
    random_state=SEED,
).fit_transform(
    penultimate_features
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)

axes[0].scatter(
    pca_xy[:, 0],
    pca_xy[:, 1],
    c=final_labels,
    s=7,
)
axes[0].set_title("PCA")

axes[1].scatter(
    umap_xy[:, 0],
    umap_xy[:, 1],
    c=final_labels,
    s=7,
)
axes[1].set_title("UMAP")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "pca_umap.png",
    dpi=170,
)
plt.show()

## 10. Local PCA + kNN

각 sample 주변의 30개 이웃만 모아서, 그 국소 영역 분산의 90%를 설명하는 데 필요한 PCA 차원 수를 셉니다.

In [ ]:
scaled_features = StandardScaler().fit_transform(
    penultimate_features
)

neighbor_indices = NearestNeighbors(
    n_neighbors=31
).fit(
    scaled_features
).kneighbors(
    return_distance=False
)

local_dimensions = []

for sample_index in range(
    min(500, len(scaled_features))
):
    local_points = scaled_features[
        neighbor_indices[sample_index]
    ]

    local_pca = PCA().fit(local_points)
    cumulative_variance = np.cumsum(
        local_pca.explained_variance_ratio_
    )

    dimension_90 = int(
        np.searchsorted(
            cumulative_variance,
            0.90,
        )
        + 1
    )
    local_dimensions.append(dimension_90)

local_dimensions = np.asarray(
    local_dimensions
)

np.save(
    NPZ_DIR / "local_pca_dim90.npy",
    local_dimensions,
)

print(
    "Local PCA dim90 mean:",
    local_dimensions.mean(),
)

## 11. Hessian top eigenvalue

Hessian 전체를 만들지 않고 Hessian-vector product와 power iteration으로 가장 큰 고유값을 근사합니다.

In [ ]:
x_hessian, y_hessian = next(iter(val_loader))
x_hessian = x_hessian[:128].to(DEVICE)
y_hessian = y_hessian[:128].to(DEVICE)


def hessian_top_eigenvalue(
    state_dict,
    iterations=12,
):
    model = SmallCNN().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()

    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    vector = [
        torch.randn_like(parameter)
        for parameter in parameters
    ]

    vector_norm = torch.sqrt(
        sum((part * part).sum() for part in vector)
    )
    vector = [
        part / vector_norm
        for part in vector
    ]

    eigenvalue = 0.0

    for _ in range(iterations):
        logits = model(x_hessian)
        loss = F.cross_entropy(
            logits,
            y_hessian,
        )

        gradients = torch.autograd.grad(
            loss,
            parameters,
            create_graph=True,
        )

        directional_gradient = sum(
            (gradient * direction).sum()
            for gradient, direction in zip(
                gradients,
                vector,
            )
        )

        hessian_vector = torch.autograd.grad(
            directional_gradient,
            parameters,
        )

        hv_norm = torch.sqrt(
            sum(
                (part * part).sum()
                for part in hessian_vector
            )
        )

        vector = [
            part.detach() / (hv_norm + 1e-12)
            for part in hessian_vector
        ]

        eigenvalue = sum(
            (direction * hv_part.detach()).sum()
            for direction, hv_part in zip(
                vector,
                hessian_vector,
            )
        ).item()

    return eigenvalue


hessian_rows = []

for condition, state_dict in [
    (
        "sgd_init",
        load_checkpoint("sgd", 0),
    ),
    (
        "sgd_final",
        load_checkpoint("sgd", EPOCHS),
    ),
    (
        "adamw_final",
        load_checkpoint("adamw", EPOCHS),
    ),
]:
    top_eigenvalue = hessian_top_eigenvalue(
        state_dict
    )

    hessian_rows.append(
        {
            "condition": condition,
            "lambda_max": top_eigenvalue,
        }
    )

hessian_df = pd.DataFrame(hessian_rows)
hessian_df.to_csv(
    CSV_DIR / "hessian_top_eigenvalue.csv",
    index=False,
)

display(hessian_df)

## 12. Weight interpolation

두 checkpoint parameter를 직선으로 섞으면서 validation loss와 accuracy를 측정합니다.

중간에서 loss가 크게 솟으면 두 solution이 직선 low-loss path로 연결되지 않는다는 뜻입니다.

In [ ]:
def interpolate_state(
    state_a,
    state_b,
    alpha,
):
    interpolated = {}

    for key in state_a:
        if torch.is_floating_point(state_a[key]):
            interpolated[key] = (
                (1.0 - alpha) * state_a[key]
                + alpha * state_b[key]
            )
        else:
            interpolated[key] = (
                state_a[key]
                if alpha < 0.5
                else state_b[key]
            )

    return interpolated


@torch.no_grad()
def interpolation_curve(
    state_a,
    state_b,
):
    model = SmallCNN().to(DEVICE)
    rows = []

    for alpha in np.linspace(0, 1, 21):
        state = interpolate_state(
            state_a,
            state_b,
            float(alpha),
        )
        model.load_state_dict(state)

        loss, accuracy = evaluate(
            model,
            val_loader,
        )

        rows.append(
            {
                "alpha": float(alpha),
                "loss": loss,
                "accuracy": accuracy,
            }
        )

    return pd.DataFrame(rows)


middle_epoch = DIAG_EPOCHS[-2]

same_run_curve = interpolation_curve(
    load_checkpoint("sgd", middle_epoch),
    load_checkpoint("sgd", EPOCHS),
)
same_run_curve["path"] = (
    "SGD middle to SGD final"
)

cross_optimizer_curve = interpolation_curve(
    load_checkpoint("sgd", EPOCHS),
    load_checkpoint("adamw", EPOCHS),
)
cross_optimizer_curve["path"] = (
    "SGD final to AdamW final"
)

interpolation_df = pd.concat(
    [
        same_run_curve,
        cross_optimizer_curve,
    ],
    ignore_index=True,
)
interpolation_df.to_csv(
    CSV_DIR / "weight_interpolation.csv",
    index=False,
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)

for path_name, group in interpolation_df.groupby("path"):
    axes[0].plot(
        group["alpha"],
        group["loss"],
        marker="o",
        label=path_name,
    )
    axes[1].plot(
        group["alpha"],
        group["accuracy"],
        marker="o",
        label=path_name,
    )

axes[0].set_title("Interpolation loss")
axes[1].set_title("Interpolation accuracy")

for axis in axes:
    axis.set_xlabel("alpha")
    axis.legend()

plt.tight_layout()
plt.savefig(
    FIG_DIR / "weight_interpolation.png",
    dpi=170,
)
plt.show()

## 13. TensorBoard와 최종 요약

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard

In [ ]:
summary = {
    "epochs": EPOCHS,
    "diagnostic_epochs": DIAG_EPOCHS,
    "sgd_final_val_accuracy": float(
        sgd_history.iloc[-1]["val_accuracy"]
    ),
    "adamw_final_val_accuracy": float(
        adamw_history.iloc[-1]["val_accuracy"]
    ),
    "local_pca_dim90_mean": float(
        local_dimensions.mean()
    ),
    "hessian": dict(
        zip(
            hessian_df["condition"],
            hessian_df["lambda_max"].astype(float),
        )
    ),
}

with open(
    SUMMARY_DIR / "experiment_summary.json",
    "w",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
    )

assert not any(DRIVE_ROOT.rglob("*.pt")), (
    "Model weight was found on Google Drive."
)

print(json.dumps(summary, indent=2))
print(
    "Storage policy passed: no .pt model weights on Drive."
)

## 결과를 읽는 권장 순서

```text
1. loss / accuracy
→ 2. gradient norm + update-to-weight
→ 3. effective rank + linear probe
→ 4. CKA
→ 5. PCA / UMAP
→ 6. local PCA
→ 7. Hessian
→ 8. interpolation
```

한 줄에 여러 연산을 몰아넣지 않고, 각 단계에서 중간 변수가 무엇인지 따라갈 수 있도록 코드를 다시 풀어서 작성했습니다.